# Transformer Attention — 구현 스켈레톤 (복원본)

> 원본 `transformer.ipynb`가 저장 누락으로 0바이트가 돼서, Week 7 대화 기록을 바탕으로 **뼈대만** 복원했어.
> 네가 채웠던 원본 셀은 어디에도 안 남아서 그대로는 못 살려 — 대신 그때 받았던 스켈레톤+가이드를 재구성했으니 TODO를 다시 채우면 돼.
> 막히면 물어봐 (힌트 모드 그대로).

## 핵심 개념 요약

**Scaled Dot-Product Attention** — `softmax(Q·Kᵀ / √d_k) · V`
1. `Q·Kᵀ` : Query와 Key 내적 → attention score
2. `/ √d_k` : 스케일링. d_k가 크면 내적값이 커져 softmax가 한쪽으로 쏠리고 gradient가 죽음 → √d_k로 나눠 안정화 (이게 'Scaled'의 의미)
3. `softmax` → 합이 1인 가중치 → `· V` 로 Value 가중합

**Multi-Head** : attention을 head개 병렬로. 각 head가 다른 관계(문법/의미)를 잡고 concat 후 출력 projection. d_model을 d_k = d_model/h 로 쪼갬.

**self vs cross** : self는 Q·K·V가 같은 시퀀스에서, cross는 Q=디코더 / K·V=인코더.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def forward(self, Q, K, V, mask=None):
        # Q, K, V: (batch, heads, seq_len, d_k)
        d_k = Q.size(-1)

        # TODO 1: scores = Q·Kᵀ / √d_k   → (batch, heads, seq_len, seq_len)
        #   힌트: torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        scores = None  # <-- 채우기

        # (선택) mask 처리: scores = scores.masked_fill(mask == 0, -1e9)

        # TODO 2: attn = softmax(scores)  (마지막 차원 기준)
        attn = None  # <-- 채우기

        # TODO 3: out = attn · V   → (batch, heads, seq_len, d_k)
        out = None   # <-- 채우기
        return out, attn

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.h = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()

    def split_heads(self, x, batch):
        # (batch, seq_len, d_model) → (batch, h, seq_len, d_k)
        # TODO: x.view(batch, -1, self.h, self.d_k).transpose(1, 2)
        return None  # <-- 채우기

    def forward(self, q, k, v, mask=None):
        batch = q.size(0)
        # TODO 1: W_q/W_k/W_v 통과 후 split_heads
        Q = None  # self.split_heads(self.W_q(q), batch)
        K = None
        V = None
        # TODO 2: attention 적용
        out, attn = None, None  # self.attention(Q, K, V, mask)
        # TODO 3: head 합치기 (batch, h, seq, d_k) → (batch, seq, d_model)
        #   out.transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        out = None
        # TODO 4: 출력 projection W_o
        out = None
        return out, attn

In [ ]:
# 동작 확인 (TODO 다 채운 뒤 실행) — 출력 shape이 입력과 같아야 정상
batch, seq, d_model, heads = 2, 5, 16, 4
x = torch.rand(batch, seq, d_model)
mha = MultiHeadAttention(d_model, heads)
out, attn = mha(x, x, x)            # self-attention
print('입력:', x.shape)            # (2, 5, 16)
print('출력:', out.shape)          # 기대: (2, 5, 16)
print('attn:', attn.shape)         # 기대: (2, 4, 5, 5)
print('attn 합:', attn.sum(-1)[0, 0])  # 전부 ≈ 1 이어야 함

## 확인 / 연결
- 출력 shape이 입력과 같은가 `(batch, seq, d_model)`
- attn 행 합이 1인가 (`attn.sum(-1) ≈ 1`)
- **오늘 QLoRA와 연결** : QLoRA가 LoRA 어댑터를 붙이는 자리가 바로 이 어텐션의 `W_q/W_k/W_v/W_o`(+MLP)야. 지금 짜는 이 모듈들이 곧 파인튜닝 대상.